# term

> A client for gateway-hosted terminals: REST lifecycle plus one websocket attachment

In [ ]:
#| default_exp term

`JupyAsyncTerminalClient` connects to terminals hosted by [rustygate](https://github.com/AnswerDotAI/rustygate) or Jupyter Server. An app can use it to provide a shell on the same machine and filesystem as its kernels, including inside a container.

The client uses `KernelApi` for terminal lifecycle requests at `/api/terminals` and a websocket for input, output and terminal controls.

In [ ]:
#| export
import json, websockets
from contextlib import suppress
from fastcore.basics import patch
from jupyasyncclient.core import KernelApi, _join_url

In [ ]:
import asyncio, os, time
from fastcore.test import test_eq
from rustygate.tools import start_gateway

## Lifecycle

Pass a gateway URL when constructing the client. To use an existing terminal, also pass its `name`. Authentication, timeout and HTTP options are the same as for `JupyAsyncKernelClient`.

`start_terminal` creates a terminal and stores its name on the client. It passes the gateway's creation options through unchanged: `argv`, `cwd`, `env`, `appendenv`, `rc`, `rows`, `cols` and `username`.

In [ ]:
#| export
class JupyAsyncTerminalClient(KernelApi):
    "One gateway terminal: REST lifecycle through the spec ops on `api`, plus one ws attachment."
    def __init__(self, base_url, name=None, token=None, headers=None, timeout=30, http_client=None, verify=True, spec=None):
        super().__init__(base_url, token=token, headers=headers, timeout=timeout, http_client=http_client, verify=verify, spec=spec)
        self.name,self._ws = name,None

    async def list_terminals(self): return await self.api.terminals.list_terms()

    async def start_terminal(self, **kw):
        "Create a terminal (gateway creation options pass through) and bind its name."
        model = await self.api.terminals.create_term(body_=kw)
        self.name = model['name']
        return model

    async def shutdown_terminal(self): return await self.api.terminals.delete_term(name=self.name)

Start a gateway and a Bash shell without startup files. A fixed prompt makes the example output predictable.

In [ ]:
BASH = ['bash', '--norc', '--noprofile', '-i']
BENV = dict(os.environ, PS1='$ ', TERM='dumb')
g = start_gateway()
tc = JupyAsyncTerminalClient(g.url)
test_eq(await tc.list_terminals(), [])
model = await tc.start_terminal(argv=BASH, env=BENV)
test_eq((await tc.list_terminals())[0]['name'], tc.name)
model

```python
{'alive': True, 'last_activity': 1789266815.656605, 'name': '1', 'path': None}
```

## The channel

`connect` opens `/terminals/websocket/{name}` and returns the server's `setup` frame. The gateway then replays available scrollback as `stdout` frames.

The websocket uses terminado's JSON-array protocol. Every frame is a JSON text list whose first element names the message type:

- The server sends `["setup", {...}]` when a client connects.
- `["stdout", text]` carries terminal output text.
- `["gap", bytes]` reports that the client fell behind the server's replay buffer (rustygate/jupygate extension).
- `["disconnect", 1, ...]` reports that the terminal exited (rustygate/jupygate include the exit code as the third element).
- The client sends `["stdin", text]` for input and `["set_size", rows, cols]` to resize the terminal.

`write` sends `stdin` text. `resize` sends `set_size`. `frames` yields each incoming frame as a parsed list. Use one `frames` iterator per connection. Multiple iterators would compete to read the same socket.

`aclose` closes this client's websocket attachment. It does not delete the terminal. Use `shutdown_terminal` to stop the terminal for all clients.

In [ ]:
#| export
@patch
async def connect(self:JupyAsyncTerminalClient):
    "Open `/terminals/websocket/{name}` and return the `setup` frame; replayed scrollback follows as `stdout` frames."
    params = dict(token=self.token) if self.token else None
    url = _join_url(self.base_url, f'/terminals/websocket/{self.name}', ws=True, params=params)
    self._ws = await websockets.connect(url, ssl=self._ws_ssl(url), additional_headers=self._headers, max_size=None)
    return json.loads(await self._ws.recv())

@patch
async def write(self:JupyAsyncTerminalClient, data:str):
    await self._ws.send(json.dumps(['stdin', data]))

@patch
async def resize(self:JupyAsyncTerminalClient, rows:int, cols:int):
    await self._ws.send(json.dumps(['set_size', rows, cols]))

@patch
async def frames(self:JupyAsyncTerminalClient):
    "Incoming terminado frames as parsed lists (`['stdout', text]`, `['gap', n]`, `['disconnect', ...]`). Ends when the ws closes normally. A broken connection raises `websockets.ConnectionClosedError`."
    async for frame in self._ws: yield json.loads(frame)

@patch
async def aclose(self:JupyAsyncTerminalClient):
    if self._ws is not None:
        with suppress(Exception): await self._ws.close()
        self._ws = None

Send a command and read its output. Frame boundaries depend on timing, so `read_until` accumulates `stdout` text until the expected pattern appears.

In [ ]:
async def read_until(fr, pat:str, timeout=10.0)->str:
    "Accumulate `stdout` frames from async iterator `fr` until `pat` appears (other frames are skipped)."
    buf = ''
    end = time.monotonic() + timeout
    while pat not in buf:
        item = await asyncio.wait_for(anext(fr), end - time.monotonic())
        if item[0] == 'stdout': buf += item[1]
    return buf

setup = await tc.connect()
test_eq(setup[0], 'setup')
fr = tc.frames()
await tc.write('echo wired $((6*7))\n')
out = await read_until(fr, 'wired 42')
out[-20:]

b'$((6*7))\r\nwired 42\r\n'

A second client can connect to the same terminal name and read its scrollback. This lets an app recover earlier output after a restart or browser refresh.

In [ ]:
tc2 = JupyAsyncTerminalClient(g.url, name=tc.name)
test_eq((await tc2.connect())[0], 'setup')
fr2 = tc2.frames()
replay = await read_until(fr2, 'wired 42')
'echo wired' in replay, 'wired 42' in replay

(True, True)

Check the resized terminal with `stty size`. Either client can shut it down by name. All attached clients then receive a `disconnect` frame.

In [ ]:
await tc.resize(50, 120)
await tc.write('stty size\n')
assert '50 120' in await read_until(fr, '50 120')
await tc2.shutdown_terminal()
item = await asyncio.wait_for(anext(fr), 10)
while item[0] == 'stdout': item = await asyncio.wait_for(anext(fr), 10)
test_eq(item[0], 'disconnect')
test_eq(await tc.list_terminals(), [])
item

{'type': 'eof', 'code': -1}

In [ ]:
#| hide
await tc.aclose()
await tc2.aclose()
g.stop()

In [ ]:
#| hide
import nbdev
nbdev.nbdev_export()